<a href="https://colab.research.google.com/github/usman-stack-322/flyrank-ml-internship-v2/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/usman-stack-322/flyrank-ml-internship-v2/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
from google.colab import userdata
from huggingface_hub import hf_hub_download
import pandas as pd

TOKEN = userdata.get("HF_TOKEN")

local_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=TOKEN
)
df_march = pd.read_parquet(local_path)

page_col = "content_hash_id"
date_col = "report_date"
impressions_col = "gsc_impressions"
clicks_col = "gsc_clicks"
position_col = "gsc_avg_position"
availability_col = "gsc_data_available"

df_march[date_col] = pd.to_datetime(df_march[date_col], errors="coerce")

print("Loaded shape:", df_march.shape)
print("Rows:", len(df_march))

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

Loaded shape: (9841378, 30)
Rows: 9841378


## 1. Unit of analysis + time window

One row = one web page (content_hash_id), for one calendar month (page × month grain).

Time window: month = 2026-03 (a mid-panel month). This avoids the final month
(June 2026 / the _sample table), which is a sealed test month reserved for
final evaluation — using it now would leak the outcome window into development.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Raw daily rows loaded for March 2026:", len(df_march))
print("Date range:", df_march[date_col].min(), "to", df_march[date_col].max())
print("Unique pages (content_hash_id):", df_march[page_col].nunique())
print("Unique clients:", df_march['client_hash_id'].nunique())


Raw daily rows loaded for March 2026: 9841378
Date range: 2026-03-01 00:00:00 to 2026-03-31 00:00:00
Unique pages (content_hash_id): 331437
Unique clients: 55


## 2. Fields: feature / label / context / excluded

Features (knowable at decision time):
- gsc_impressions — this month's observed impressions
- gsc_clicks — this month's observed clicks
- ctr — clicks/impressions within the same month (derived)
- gsc_avg_position — this month's average SERP position
- page_age_days — days since page first seen (placeholder here, see limitations)

Label/proxy: content opportunity score — pages with high impressions but
below-median CTR (seen a lot, clicked little) = higher opportunity.

Context (identifiers, not features): content_hash_id (page), client_hash_id, report_date/month

Excluded: any gsc_impressions/gsc_clicks/gsc_avg_position values from AFTER the
decision month, all ga4_* engagement fields (out of scope for this lane), and
anything computed directly from the label. Reason: these wouldn't exist yet at
the moment we'd need to make the prediction — including them would leak future
information into training.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Feature source columns confirmed:")
print(" impressions:", impressions_col, "| clicks:", clicks_col, "| position:", position_col, "| availability:", availability_col)
print(df_march[[impressions_col, clicks_col, position_col, availability_col]].dtypes)


Feature source columns confirmed:
 impressions: gsc_impressions | clicks: gsc_clicks | position: gsc_avg_position | availability: gsc_data_available
gsc_impressions         int64
gsc_clicks              int64
gsc_avg_position      float64
gsc_data_available       bool
dtype: object


## 3. Verify it with queries (grain, counts, missing values, windows)

Three verification queries: grain, coverage (row count + date span), availability
(IS TRUE filter). Then a five-feature frame, then the deliberate leakage trap —
added, observed, and removed.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb, numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

con = duckdb.connect()

page_month = df_march.groupby(page_col, as_index=False).agg({
    impressions_col: "sum",
    clicks_col: "sum",
    position_col: "mean"
})
page_month["ctr"] = page_month[clicks_col] / page_month[impressions_col].replace(0, np.nan)
con.register("page_month", page_month)

grain = con.execute(f"""
    SELECT COUNT(*) AS total_rows, COUNT(DISTINCT {page_col}) AS distinct_pages,
           COUNT(*) - COUNT(DISTINCT {page_col}) AS duplicate_rows
    FROM page_month
""").df()
print("Query 1 - Grain check:")
print(grain)
assert grain.loc[0, "duplicate_rows"] == 0, "Grain broken: duplicate page rows"


Query 1 - Grain check:
   total_rows  distinct_pages  duplicate_rows
0      331437          331437               0


In [5]:
print("Query 2 - Coverage:")
print("Page-month rows:", len(page_month))
print("Date span (source daily rows):", df_march[date_col].min(), "to", df_march[date_col].max())

Query 2 - Coverage:
Page-month rows: 331437
Date span (source daily rows): 2026-03-01 00:00:00 to 2026-03-31 00:00:00


In [6]:
con.register("df_march_reg", df_march)
before = con.execute("SELECT COUNT(*) AS n FROM df_march_reg").df()
after = con.execute(f"SELECT COUNT(*) AS n FROM df_march_reg WHERE {availability_col} IS TRUE").df()
print("Query 3 - Availability:")
print("Before:", before.loc[0,'n'], "| After IS TRUE filter:", after.loc[0,'n'])

Query 3 - Availability:
Before: 9841378 | After IS TRUE filter: 3611061


In [7]:
page_month["page_age_days"] = 365
feature_cols = [clicks_col, impressions_col, "ctr", position_col, "page_age_days"]
print("Feature frame (5 features):")
print(page_month[[page_col] + feature_cols].head())

Feature frame (5 features):
            content_hash_id  gsc_clicks  gsc_impressions  ctr  \
0  content_000005d4ced12088           0               86  0.0   
1  content_00001e488b74b799           0                0  NaN   
2  content_00007bd2985b77c3           0               47  0.0   
3  content_00008950670cb6b5           0                0  NaN   
4  content_0000a348850eb1fc           0                0  NaN   

   gsc_avg_position  page_age_days  
0         72.854861            365  
1               NaN            365  
2          5.269565            365  
3               NaN            365  
4               NaN            365  


In [8]:
work = page_month.copy()
work = work[work[impressions_col] > 0].copy()

impr_threshold = work[impressions_col].quantile(0.5)
ctr_threshold = work["ctr"].quantile(0.5)

work["opportunity_label"] = (
    (work[impressions_col] >= impr_threshold) &
    (work["ctr"] <= ctr_threshold)
).astype(int)

print("Label distribution:")
print(work["opportunity_label"].value_counts())

X = work[feature_cols].fillna(0)
y = work["opportunity_label"]
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
honest_auc = roc_auc_score(yte, LogisticRegression(max_iter=1000).fit(Xtr, ytr).predict_proba(Xte)[:, 1])
print(f"\nHonest AUC (no leakage): {honest_auc:.4f}")

Label distribution:
opportunity_label
0    149172
1     27566
Name: count, dtype: int64

Honest AUC (no leakage): 0.9996


In [9]:
work["leaky_col"] = work["opportunity_label"] * 1.0 + np.random.normal(0, 0.01, len(work))

Xl = work[feature_cols + ["leaky_col"]].fillna(0)
Xtrl, Xtel, ytrl, ytel = train_test_split(Xl, y, test_size=0.3, random_state=42, stratify=y)
leaky_auc = roc_auc_score(ytel, LogisticRegression(max_iter=1000).fit(Xtrl, ytrl).predict_proba(Xtel)[:, 1])
print(f"Leaky AUC (with label-derived column): {leaky_auc:.4f}  <- trap firing, jumps toward 1.0")
print(f"Jump: {leaky_auc - honest_auc:+.4f}")

Leaky AUC (with label-derived column): 1.0000  <- trap firing, jumps toward 1.0
Jump: +0.0004


In [10]:
del work["leaky_col"]
print("leaky_col removed.")
print(f"Final honest number to report: {honest_auc:.4f}")
print(f"(Not {leaky_auc:.4f} — that number was an artifact of the label leaking into a feature.)")

leaky_col removed.
Final honest number to report: 0.9996
(Not 1.0000 — that number was an artifact of the label leaking into a feature.)


## 4. Data limits

- Single month only (2026-03): no seasonal pattern is captured, this is a snapshot.
- page_age_days is a placeholder (365 for all rows) — no first-seen date column
  was found in fact_content_daily_performance; a real value would need a join
  against dim_content.
- Multiple clients share this table; the opportunity score here is not
  client-normalized, so it may be biased toward high-traffic clients.
- This slice is search-performance only — no conversions/revenue, so
  "opportunity" is a proxy, not a business-validated outcome.
- Results are observed/directional and meant for decision support, not causal proof.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Clients represented in this slice:", df_march['client_hash_id'].nunique())
print(page_month['page_age_days'].describe())


Clients represented in this slice: 55
count    331437.0
mean        365.0
std           0.0
min         365.0
25%         365.0
50%         365.0
75%         365.0
max         365.0
Name: page_age_days, dtype: float64


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.